# Load the data set

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv("../data/training.csv")

In [3]:
print(df.shape)

(640840, 10)


In [4]:
print(df.columns)

Index(['Unnamed: 0', 'store_ID', 'day_of_week', 'date', 'nb_customers_on_day',
       'open', 'promotion', 'state_holiday', 'school_holiday', 'sales'],
      dtype='str')


In [5]:
print(df.dtypes)

Unnamed: 0             int64
store_ID               int64
day_of_week            int64
date                     str
nb_customers_on_day    int64
open                   int64
promotion              int64
state_holiday            str
school_holiday         int64
sales                  int64
dtype: object


In [6]:
print(df.isna())

        Unnamed: 0  store_ID  day_of_week   date  nb_customers_on_day   open  \
0            False     False        False  False                False  False   
1            False     False        False  False                False  False   
2            False     False        False  False                False  False   
3            False     False        False  False                False  False   
4            False     False        False  False                False  False   
...            ...       ...          ...    ...                  ...    ...   
640835       False     False        False  False                False  False   
640836       False     False        False  False                False  False   
640837       False     False        False  False                False  False   
640838       False     False        False  False                False  False   
640839       False     False        False  False                False  False   

        promotion  state_holiday  schoo

In [7]:
df.head()

,Unnamed: 0,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
0,425390,366,4,2013-04-18,517,1,0,0,0,4422
1,291687,394,6,2015-04-11,694,1,0,0,0,8297
2,411278,807,4,2013-08-29,970,1,1,0,0,9729
3,664714,802,2,2013-05-28,473,1,1,0,0,6513
4,540835,726,4,2013-10-10,1068,1,1,0,0,10882


In [8]:
print(df.isna().sum())

Unnamed: 0             0
store_ID               0
day_of_week            0
date                   0
nb_customers_on_day    0
open                   0
promotion              0
state_holiday          0
school_holiday         0
sales                  0
dtype: int64


In [9]:
print("state holiday:", df["state_holiday"].unique())
print("number of customers per day:", df["nb_customers_on_day"].unique())
print("open:", df["open"].unique())

state holiday: <StringArray>
['0', 'a', 'c', 'b']
Length: 4, dtype: str
number of customers per day: [ 517  694  970 ... 4630 3713 4003]
open: [1 0]


In [10]:
df[df["sales"] == 0]

,Unnamed: 0,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
6,600327,659,7,2014-06-08,0,0,0,0,0,0
10,561067,273,7,2014-10-05,0,0,0,0,0,0
18,409022,767,7,2013-01-27,0,0,0,0,0,0
34,605423,534,7,2014-06-08,0,0,0,0,0,0
35,231682,514,7,2014-03-09,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
640807,381567,167,7,2014-07-27,0,0,0,0,0,0
640812,49811,870,3,2014-07-23,0,0,0,0,1,0
640814,556209,650,7,2014-03-23,0,0,0,0,0,0
640834,304137,329,7,2013-09-15,0,0,0,0,0,0


In [11]:
df[df["nb_customers_on_day"] == 0]

,Unnamed: 0,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
6,600327,659,7,2014-06-08,0,0,0,0,0,0
10,561067,273,7,2014-10-05,0,0,0,0,0,0
18,409022,767,7,2013-01-27,0,0,0,0,0,0
34,605423,534,7,2014-06-08,0,0,0,0,0,0
35,231682,514,7,2014-03-09,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
640807,381567,167,7,2014-07-27,0,0,0,0,0,0
640812,49811,870,3,2014-07-23,0,0,0,0,1,0
640814,556209,650,7,2014-03-23,0,0,0,0,0,0
640834,304137,329,7,2013-09-15,0,0,0,0,0,0


# Find the anomalous row

In [12]:
anomaly = df[(df['sales'] == 0) & (df['nb_customers_on_day'] > 0)]
print(anomaly)

        Unnamed: 0  store_ID  day_of_week        date  nb_customers_on_day  \
352744      319242       948            4  2013-04-25                    5   

        open  promotion state_holiday  school_holiday  sales  
352744     1          1             0               0      0  


# Data Cleaning

### Drop the row identifier

In [13]:
df.drop(columns=['Unnamed: 0'], inplace=True)
print('Dropped row-ID column. New shape:', df.shape)

Dropped row-ID column. New shape: (640840, 9)


### Fix `date` data types , extract useful features, drop the original `date` column

In [14]:
df["date"] = pd.to_datetime(df["date"])
print(df["date"].dtype)

df['year']        = df['date'].dt.year
df['month']       = df['date'].dt.month
df['day']         = df['date'].dt.day
df['week']        = df['date'].dt.isocalendar().week.astype(int)
df['is_weekend']  = df['date'].dt.dayofweek.isin([5, 6]).astype(int)
df = df.drop(columns=['date'])

print(df.head())

datetime64[us]
   store_ID  day_of_week  nb_customers_on_day  open  promotion state_holiday  \
0       366            4                  517     1          0             0   
1       394            6                  694     1          0             0   
2       807            4                  970     1          1             0   
3       802            2                  473     1          1             0   
4       726            4                 1068     1          1             0   

   school_holiday  sales  year  month  day  week  is_weekend  
0               0   4422  2013      4   18    16           0  
1               0   8297  2015      4   11    15           1  
2               0   9729  2013      8   29    35           0  
3               0   6513  2013      5   28    22           0  
4               0  10882  2013     10   10    41           0  


### for column `state_holiday ` Creates a binary column for each category, drops holiday_0 to avoid the dummy variable trap (multicollinearity), 

In [15]:
df = pd.get_dummies(df, columns=['state_holiday'], prefix='holiday', drop_first=True)
print(df.head())

   store_ID  day_of_week  nb_customers_on_day  open  promotion  \
0       366            4                  517     1          0   
1       394            6                  694     1          0   
2       807            4                  970     1          1   
3       802            2                  473     1          1   
4       726            4                 1068     1          1   

   school_holiday  sales  year  month  day  week  is_weekend  holiday_a  \
0               0   4422  2013      4   18    16           0      False   
1               0   8297  2015      4   11    15           1      False   
2               0   9729  2013      8   29    35           0      False   
3               0   6513  2013      5   28    22           0      False   
4               0  10882  2013     10   10    41           0      False   

   holiday_b  holiday_c  
0      False      False  
1      False      False  
2      False      False  
3      False      False  
4      False      Fals

In [16]:
df[(df["open"] == 0) & (df["sales"] > 0)]

,store_ID,day_of_week,nb_customers_on_day,open,promotion,school_holiday,sales,year,month,day,week,is_weekend,holiday_a,holiday_b,holiday_c


In [17]:
df[(df["open"] == 1) & (df["sales"] == 0)]

,store_ID,day_of_week,nb_customers_on_day,open,promotion,school_holiday,sales,year,month,day,week,is_weekend,holiday_a,holiday_b,holiday_c
54379,339,3,0,1,0,0,0,2013,1,30,5,0,False,False,False
61554,1039,3,0,1,0,0,0,2013,7,10,28,0,False,False,False
63404,983,5,0,1,0,0,0,2014,1,17,3,0,False,False,False
82644,387,4,0,1,0,1,0,2014,7,24,30,0,False,False,False
117219,762,4,0,1,0,0,0,2013,1,17,3,0,False,False,False
137903,303,4,0,1,0,1,0,2014,7,24,30,0,False,False,False
163742,25,3,0,1,0,0,0,2014,2,12,7,0,False,False,False
180912,835,4,0,1,0,0,0,2014,9,11,37,0,False,False,False
188539,835,3,0,1,0,0,0,2014,9,10,37,0,False,False,False
209801,548,5,0,1,1,1,0,2014,9,5,36,0,False,False,False


In [18]:
df = df[df["open"] == 1]
print(df.shape)

(532016, 15)


In [19]:
open_with_customers =  df[(df['open'] == 1) & (df['nb_customers_on_day'] > 0)]
total = len(open_with_customers)
anomalies = len(open_with_customers[open_with_customers['sales'] == 0])
pct = (anomalies / total) * 100
print(f'Total open days with customers: {total}')
print(f'Days with sales = 0:            {anomalies}')
print(f'Anomaly percentage:             {pct:.4f}%')
print(df.shape)

Total open days with customers: 531986
Days with sales = 0:            1
Anomaly percentage:             0.0002%
(532016, 15)


# Model 1: linear regression

define the features and target

In [22]:
X = df.drop(columns=['sales'])
y = df['sales']

train and test split

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')

Train size: 399012 | Test size: 133004


train linear regression

In [32]:
lr = LinearRegression()
lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


evaluate

In [33]:
y_pred = lr.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'\n=== Linear Regression Results ===')
print(f'RMSE : {rmse:.2f}')
print(f'R²   : {r2:.4f}')


=== Linear Regression Results ===
RMSE : 1593.77
R²   : 0.7366
